In [1]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TABLA ALEMANIA — Limpieza completa paso a paso
# ------------------------------------------------------------
# 🌍 Este dataset tiene información de todos los municipios de
#    Alemania: nombres, códigos postales, población, coordenadas...
#    Es como un directorio telefónico gigante del país, pero con
#    datos de cada pueblo y ciudad. Vamos a dejarlo limpio y listo
#    para trabajar, paso a paso y con semáforos para ver qué está bien.
print("-" * 60)
print("INICIO LIMPIEZA: germany_municipalities")
print("-" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
# 📂 Abrimos el CSV. Este archivo tiene un detalle: usa encoding
#    "utf-8-sig", que es como un UTF-8 pero con un "sello" al inicio
#    (BOM) que meten algunos programas de Windows. Si no lo usamos,
#    la primera columna aparecería con caracteres basura al inicio.
# Igual que antes: leemos todo como texto y no dejamos que Pandas
# invente nulos por su cuenta. Primero ver el archivo tal cual.
df_de = pd.read_csv(
    "germany_municipalities_jun_2025.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_de.shape[0]} filas x {df_de.shape[1]} columnas")
print(f"    → Es como abrir un directorio telefónico de todo un país:")
print(f"      {df_de.shape[0]:,} municipios (filas), cada uno con {df_de.shape[1]} datos (columnas).")
print(f"    Columnas: {df_de.columns.tolist()}")

# - Guardamos los nombres originales para después poder auditar
#   qué se renombró y qué se eliminó. Es como guardar la foto
#   del "antes" para compararla con el "después" al final.
cols_originales = df_de.columns.tolist()

# ------------------------------------------------------------
# 2. RENAME
# ------------------------------------------------------------
# 🏷️ Los nombres de las columnas vienen con paréntesis y mezcla de
#    inglés/alemán: "municipality_name_(gemeindename)". Eso es como
#    una etiqueta escrita por dos personas distintas: funciona, pero
#    es incómoda. Las renombramos a algo más limpio y en un solo idioma.
df_de = df_de.rename(columns={
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
})
print(f"\n[2] 🏷️ Rename hecho. Columnas nuevas: {df_de.columns.tolist()}")
print(f"    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.")
print(f"      Más corto, más claro, más fácil de escribir en el código.")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS Y VACÍOS
# ------------------------------------------------------------
# 🩺 Antes de operar, diagnosticamos. Contamos cuántos vacíos ("")
#    y cuántos nulos reales (NaN) hay en cada columna, y lo mostramos
#    ordenado de la más problemática a la más limpia.
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_de.dtypes,
    "vacios_str":      (df_de == "").sum(),
    "nan_reales":      df_de.isnull().sum(),
    "total_problemas": ((df_de == "") | df_de.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (
    reporte_nulos["total_problemas"] / len(df_de) * 100
).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)
print(reporte_nulos.to_string())

# 🚦 Y ahora el semáforo: un color por columna según qué tan grave es.
#    Es como el tablero de un auto: verde = todo bien, rojo = peligro.
print("\nSemáforo por columna:")
for col in reporte_nulos.index:
    pct = reporte_nulos.loc[col, "pct_del_total"]
    if   pct == 0:   icono = "✅"  # Perfecto, sin problemas.
    elif pct < 5:    icono = "🟢"  # Casi limpio, muy poquitos vacíos.
    elif pct < 30:   icono = "🟡"  # Ojo, hay que prestarle atención.
    elif pct < 70:   icono = "🟠"  # Bastante sucio, hay que trabajarlo.
    else:            icono = "🔴"  # Casi vacío, cuidado con esta columna.
    print(f"  {icono}  {col:<25} → {pct:>6.2f}%  "
          f"({reporte_nulos.loc[col, 'total_problemas']} filas)")

# Ahora sí, convertimos los vacíos en nulos reales. Es como marcar
# con resaltador las casillas en blanco, para encontrarlas fácil.
df_de = df_de.replace({"": np.nan})
print("\n[2.5] 🔄 Vacíos convertidos a NaN reales.")

# ------------------------------------------------------------
# 2.75 ESTANDARIZAR TEXTO
# ------------------------------------------------------------
# 🔤 Los textos (estados, distritos, nombres de municipios) vienen
#    con mayúsculas, minúsculas, espacios extra y hasta comillas.
#    Es como si cada quien hubiera escrito las etiquetas a su manera.
#    Los unificamos para poder agrupar y comparar sin sorpresas.
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

text_cols = [
    "state",
    "district",
    "regional_type",
    "municipality_name",
    "municipality_type",
]

# Primero nos aseguramos de que sean texto puro (por si algún número
# se coló como texto en algún lado). Es como asegurarnos de tener
# todas las piezas del mismo material antes de pintarlas.
for col in text_cols:
    df_de[col] = df_de[col].astype(str)

# ✂️ A minúsculas y sin espacios en las orillas: "  Bavaria " → "bavaria".
for col in text_cols:
    df_de[col] = df_de[col].str.lower().str.strip()

# 🧹 Quitamos las comillas de las orillas: '"berlin"' → 'berlin'.
#    Es como pelar una naranja antes de comérsela.
for col in text_cols:
    df_de[col] = df_de[col].str.strip('"')

# 🧽 Colapsamos espacios dobles: "baden   wurttemberg" → "baden wurttemberg".
for col in text_cols:
    df_de[col] = df_de[col].str.replace(r"\s+", " ", regex=True)

# Cuando convertimos a str, los NaN se volvieron el texto "nan".
# Aquí los regresamos a nulos reales. Es como deshacer un disfraz.
df_de[text_cols] = df_de[text_cols].replace({"nan": np.nan, "": np.nan})

print("Estandarización aplicada a:")
for col in text_cols:
    print(f"  · {col}")

print(f"\nEjemplo 'state' únicos: {df_de['state'].dropna().unique()[:5]}")
print(f"Ejemplo 'district' únicos: {df_de['district'].dropna().unique()[:5]}")
print(f"  → Antes había 'Bavaria', 'BAVARIA', '  bavaria'; ahora todos son 'bavaria'.")

# ------------------------------------------------------------
# 3. NULOS LÓGICOS / VÁLIDOS
# ------------------------------------------------------------
# 🤔 Aquí un matiz importante: no todos los nulos son errores.
#    Un municipio sin "tipo" probablemente ES un municipio normal,
#    solo que nadie lo clasificó explícitamente. Igual con el
#    "regional_type": si no tiene designación, es que no aplica.
antes_nulos_mt = df_de["municipality_type"].isnull().sum()
antes_nulos_rt = df_de["regional_type"].isnull().sum()

df_de["municipality_type"] = df_de["municipality_type"].fillna("Municipio")
df_de["regional_type"]     = df_de["regional_type"].fillna("Sin Designación")

print(f"\n[3] 🤔 Nulos lógicos rellenados:")
print(f"    municipality_type: {antes_nulos_mt} filas → 'Municipio'")
print(f"    regional_type:     {antes_nulos_rt} filas → 'Sin Designación'")
print(f"  → Es como cuando en un formulario dejas 'tipo de documento' en blanco:")
print(f"    se asume que es el más común. Aquí, el tipo más común es 'Municipio'.")

# ------------------------------------------------------------
# 4. BUGS CRÍTICOS → dropna
# ------------------------------------------------------------
# 🫀 Hay datos que son como el "corazón" de una fila: si faltan,
#    la fila pierde sentido. Sin código postal o sin coordenadas,
#    un municipio no nos sirve para nada geográfico. Los eliminamos.
antes = len(df_de)
criticas = ["postal_code", "longitude", "latitude"]
print(f"\n[4] 🫀 Nulos en columnas críticas antes del dropna:")
print(df_de[criticas].isnull().sum().to_string())

df_de = df_de.dropna(subset=criticas)
print(f"    📉 Filas eliminadas: {antes - len(df_de)}")
print(f"    ✅ Filas restantes:  {len(df_de)}")
print(f"  → Es como tirar direcciones incompletas: si no hay código postal")
print(f"    ni coordenadas, el municipio no se puede ubicar en el mapa.")

# ------------------------------------------------------------
# 5. PARSEAR COORDENADAS
# ------------------------------------------------------------
# 🗺️ Las coordenadas vienen como texto, a veces con coma decimal
#    europea ("13,405" en vez de "13.405") y a veces con valores
#    rotos como "1340500" (un decimal perdido). Nuestra función
#    limpia todo eso: convierte comas en puntos y, si el número
#    es demasiado grande para ser una coordenada real, lo divide
#    entre 10 hasta que quede en el rango válido.
def parse_coord(value, tipo):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    s = str(value).strip().replace(",", ".")
    try:
        num = float(s)
    except ValueError:
        return np.nan
    # Si un decimal se perdió al exportar, el número queda enorme.
    # Dividir entre 10 hasta que encaje en el rango correcto
    # es como "bajarle el volumen" al número hasta que suene bien.
    if tipo == "longitude":
        while num > 180:
            num /= 10
    else:
        while num > 90:
            num /= 10
    return num

df_de["longitude"] = df_de["longitude"].apply(lambda v: parse_coord(v, "longitude"))
df_de["latitude"]  = df_de["latitude"].apply(lambda v: parse_coord(v, "latitude"))

coords_rotas = df_de["longitude"].isnull().sum() + df_de["latitude"].isnull().sum()
print(f"\n[5] 🗺️ Coordenadas parseadas")
print(f"    Longitud  → min: {df_de['longitude'].min():.4f} | max: {df_de['longitude'].max():.4f}")
print(f"    Latitud   → min: {df_de['latitude'].min():.4f} | max: {df_de['latitude'].max():.4f}")
print(f"    ⚠️  Coords que no se pudieron parsear: {coords_rotas}")
print(f"  → Alemania está más o menos entre longitud 5.5 y 15.5, y latitud")
print(f"    47 y 55.5. Si los min/max caen dentro de esos rangos, todo bien.")

# ------------------------------------------------------------
# 6. CONVERTIR NUMÉRICOS
# ------------------------------------------------------------
# 🔢 Estas columnas deben ser números, pero llegaron como texto
#    porque al inicio le dijimos a Pandas que leyera todo como str.
#    Ahora las convertimos a número. Si alguna celda tiene texto
#    basura que no se puede convertir, queda como nulo (coerce).
for col in ["area_km2", "population_total", "male", "female"]:
    antes_nulos = df_de[col].isnull().sum()
    df_de[col] = pd.to_numeric(df_de[col], errors="coerce")
    despues_nulos = df_de[col].isnull().sum()
    print(f"\n[6] 🔢 {col}: {antes_nulos} → {despues_nulos} nulos tras to_numeric")
print(f"  → Si algún nulo apareció de la nada, es porque esa celda tenía")
print(f"    texto que no se podía leer como número. Mejor perder ese dato")
print(f"    que arrastrar un valor mal calculado.")

# ------------------------------------------------------------
# 7. POBLACIÓN según tipo de municipio
# ------------------------------------------------------------
# 💡 Hay municipios especiales en Alemania llamados "gemeindefreies
#    Gebiet" (área libre de municipio): son zonas sin población real,
#    como bosques o terrenos militares. Para esos, la población
#    nula ES correcta: debería ser 0. No son datos faltantes.
mask_no_habitado = df_de["municipality_type"].str.contains("gemfr", case=False, na=False)
print(f"\n[7] 💡 Municipios 'gemfr. gebiet' (sin población real): {mask_no_habitado.sum()}")
print(f"  → Son áreas deshabitadas por definición, no errores del sistema.")

# Solo miramos los habitados: esos sí deberían tener población.
mask_habitado = ~mask_no_habitado
poblacion_nula_habitado = df_de.loc[mask_habitado, "population_total"].isnull().sum()
print(f"    Municipios habitados con población nula: {poblacion_nula_habitado}")

# Rellenamos con 0 solo los no habitados (correcto).
df_de.loc[mask_no_habitado, "population_total"] = (
    df_de.loc[mask_no_habitado, "population_total"].fillna(0)
)
# Y eliminamos los habitados con población nula: ahí sí hay un error.
antes = len(df_de)
df_de = df_de.dropna(subset=["population_total"])
print(f"    📉 Filas eliminadas por población nula: {antes - len(df_de)}")

# ------------------------------------------------------------
# 8. DENSIDAD recalculada + drop de area_km2
# ------------------------------------------------------------
# 🧮 La columna "per_km2" (densidad original) puede venir mal calculada
#    o traer valores raros. Mejor la recalculamos nosotros mismos a
#    partir de población y área. Es como verificar una cuenta ajena
#    con calculadora propia.
#
# ⚠️ Ojo con la división por cero: si algún municipio tiene área 0
#    (probablemente un error), la división daría infinito. Lo tratamos.
area_cero = (df_de["area_km2"] == 0).sum()
print(f"\n[8] 🧮 Densidad recalculada:")
print(f"    Filas con area_km2 = 0 (dividirían por 0): {area_cero}")

df_de["population_density"] = df_de["population_total"] / df_de["area_km2"]
# Los infinitos se convierten en nulos: mejor un hueco que un valor absurdo.
df_de["population_density"] = df_de["population_density"].replace(
    [np.inf, -np.inf], np.nan
)

# Después de recalcular la densidad, ya no necesitamos area_km2.
# Es como usar una escalera para subir al techo y luego tirarla:
# ya cumplió su función, no hay por qué cargarla.
df_de = df_de.drop(columns=["area_km2"])
print(f"    ✂️ Columna 'area_km2' eliminada.")
print(f"  → La usamos para calcular la densidad, y ahora ya no la necesitamos.")

# ------------------------------------------------------------
# 9. AUDITORÍA FINAL — comparativa antes/después
# ------------------------------------------------------------
# 🧾 Repaso final antes de guardar. Como el chef que prueba el plato
#    antes de sacarlo al cliente: verificamos que todo esté en orden.
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)
print(f"Filas finales: {len(df_de):,}")
print(f"\n🔍 Nulos restantes por columna:")
print(df_de.isnull().sum().sort_values(ascending=False))
print(f"\n🔑 Duplicados: {df_de.duplicated().sum()}")
print(f"  → Si es 0, cada fila es única. Si es >0, hay que investigar.")
print(f"\n👀 Primeras 3 filas:")
print(df_de.head(3).to_string())
print(f"\n📋 Tipos de datos:")
print(df_de.dtypes)

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos todo limpio en un CSV nuevo. Es como sacar una foto
#    del trabajo terminado para no repetir la limpieza después.
df_de.to_csv("germany_municipalities_clean.csv", index=False)
print(f"\n✅ Guardado: germany_municipalities_clean.csv ({len(df_de)} filas)")

# ------------------------------------------------------------
# 11. VALIDACIONES POST-GUARDADO
# ------------------------------------------------------------
# 📸 Y ahora la parte más importante: volvemos a abrir el archivo
#    que acabamos de guardar y lo revisamos todo de nuevo.
#    Es como mirar la foto que acabamos de tomar por si salió borrosa.
print("\n" + "=" * 60)
print("🔍 VALIDACIONES POST-GUARDADO")
print("=" * 60)

# - Recargar el CSV guardado para verificar integridad
df_check = pd.read_csv("germany_municipalities_clean.csv")

# ------------------------------------------------------------
# 11.1 Resumen general
# ------------------------------------------------------------
print(f"\n[11.1] 📊 RESUMEN GENERAL")
print(f"    Filas guardadas:    {df_check.shape[0]:,}")
print(f"    Columnas guardadas: {df_check.shape[1]}")
print(f"    Tamaño en memoria:  {df_check.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"  → Si las filas coinciden con las que teníamos antes de guardar,")
print(f"    el archivo se escribió bien. Si no, algo se perdió en el camino.")

# ------------------------------------------------------------
# 11.2 Columnas finales (orden exacto)
# ------------------------------------------------------------
print(f"\n[11.2] 📜 COLUMNAS FINALES ({df_check.shape[1]}):")
for i, col in enumerate(df_check.columns, 1):
    print(f"    {i:>2}. {col:<25} ({df_check[col].dtype})")
print(f"  → Este es el inventario exacto de lo que hay en la bodega final.")

# ------------------------------------------------------------
# 11.3 Columnas eliminadas
# ------------------------------------------------------------
print(f"\n[11.3] ✂️ COLUMNAS ELIMINADAS:")
# Aquí comparamos lo original con lo final, ajustando por los renombres:
# una columna renombrada no es "eliminada", solo cambió de nombre.
cols_renombradas = {
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
}

# - Ajustando las que fueron renombradas (no son "eliminadas")
cols_originales_efectivas = [
    cols_renombradas.get(c, c) for c in cols_originales
]
cols_eliminadas = [
    c for c in cols_originales_efectivas if c not in df_check.columns
]

if cols_eliminadas:
    for col in cols_eliminadas:
        print(f"    ✗ {col}")
else:
    print("    (ninguna)")

print(f"\n    Total columnas originales:  {len(cols_originales)}")
print(f"    Total columnas renombradas: {len(cols_renombradas)}")
print(f"    Total columnas eliminadas:  {len(cols_eliminadas)}")
print(f"    Total columnas finales:     {len(df_check.columns)}")
print(f"  → Si renombradas + eliminadas + conservadas = originales,")
print(f"    no se nos escapó ninguna columna por el camino.")

# ------------------------------------------------------------
# 11.4 Verificación de nulos restantes
# ------------------------------------------------------------
print(f"\n[11.4] 🔍 NULOS EN EL CSV GUARDADO:")
nulos_check = df_check.isnull().sum()
nulos_check = nulos_check[nulos_check > 0].sort_values(ascending=False)

if len(nulos_check) == 0:
    print("    ✅ Sin nulos restantes")
    print("    → La tabla quedó limpia como un vidrio recién lavado.")
else:
    for col, n in nulos_check.items():
        pct = n / len(df_check) * 100
        # 🚦 Semáforo también aquí, para ver rápido qué columnas
        #    conservan huecos y cuánta gravedad tiene.
        if   pct < 5:    icono = "🟢"
        elif pct < 30:   icono = "🟡"
        elif pct < 70:   icono = "🟠"
        else:            icono = "🔴"
        print(f"    {icono} {col:<25} → {n:>6} filas ({pct:>6.2f}%)")

# ------------------------------------------------------------
# 11.5 Verificación de duplicados
# ------------------------------------------------------------
dups = df_check.duplicated().sum()
print(f"\n[11.5] 🔑 DUPLICADOS EN EL CSV GUARDADO: {dups}")
if dups == 0:
    print("    ✅ Sin duplicados")
    print("    → Cada fila es única, no hay municipios repetidos.")
else:
    print(f"    ⚠️  {dups} filas duplicadas")
    print(f"    → Habría que investigar por qué se repitieron.")

# ------------------------------------------------------------
# 11.6 Verificación de columnas que NO deben estar
# ------------------------------------------------------------
print(f"\n[11.6] 🚫 VERIFICACIÓN DE COLUMNAS ELIMINADAS:")
# Esta verificación es como revisar que la basura que sacamos
# efectivamente NO se quedó dentro de la casa.
cols_prohibidas = ["area_km2", "per_km2", "municipality_name_(gemeindename)",
                   "municipality_designation", "regional_designation",
                   "state_(land)", "district_(kreis)"]
encontradas = [c for c in cols_prohibidas if c in df_check.columns]
if encontradas:
    print(f"    ⚠️  Columnas que NO deberían estar: {encontradas}")
else:
    print("    ✅ Ninguna columna eliminada/renombrada quedó en el CSV")
    print("    → La limpieza fue efectiva: lo que sacamos, se quedó fuera.")

# ------------------------------------------------------------
# 11.7 Verificación de tipos de datos
# ------------------------------------------------------------
print(f"\n[11.7] 📋 TIPOS DE DATOS:")
print(df_check.dtypes.to_string())
print(f"  → Los tipos numéricos deberían ser float/int, y los textos object.")
print(f"    Si algún número aparece como object, hay texto escondido adentro.")

# ------------------------------------------------------------
# 11.8 Verificación de rangos de coordenadas
# ------------------------------------------------------------
# 🗺️ Verificación geográfica: si las coordenadas caen todas dentro
#    del "rectángulo" de Alemania, todo tiene sentido. Si alguna
#    cae en el océano o en otro país, algo se rompió al parsear.
print(f"\n[11.8] 🗺️ RANGOS DE COORDENADAS:")
if "longitude" in df_check.columns and "latitude" in df_check.columns:
    print(f"    Longitud  → min: {df_check['longitude'].min():>8.4f} | "
          f"max: {df_check['longitude'].max():>8.4f}")
    print(f"    Latitud   → min: {df_check['latitude'].min():>8.4f} | "
          f"max: {df_check['latitude'].max():>8.4f}")

    # - Validar rango geográfico de Alemania (aprox.)
    lon_ok = df_check["longitude"].between(5.5, 15.5).all()
    lat_ok = df_check["latitude"].between(47.0, 55.5).all()
    print(f"    ¿Longitudes dentro de Alemania (5.5–15.5)?:  {lon_ok}")
    print(f"    ¿Latitudes dentro de Alemania (47.0–55.5)?: {lat_ok}")
    if lon_ok and lat_ok:
        print(f"  → ✅ Todas las coordenadas caen dentro de Alemania.")
        print(f"    Ningún municipio apareció flotando en el Atlántico.")
    else:
        print(f"  → ⚠️  Hay coordenadas fuera de Alemania: hay que revisar el parseo.")
else:
    print("    ⚠️  Columnas de coordenadas no encontradas")

# ------------------------------------------------------------
# 11.9 Verificación de densidad
# ------------------------------------------------------------
# 📊 La densidad de población debería ser un número razonable:
#    ningún municipio tiene millones de personas por km², ni
#    debería haber densidades negativas (sería absurdo).
print(f"\n[11.9] 📊 VERIFICACIÓN DE DENSIDAD:")
if "population_density" in df_check.columns:
    print(f"    Densidad → min: {df_check['population_density'].min():>10.2f} | "
          f"max: {df_check['population_density'].max():>10.2f}")
    print(f"    Densidad → media: {df_check['population_density'].mean():>10.2f}")
    nulos_dens = df_check["population_density"].isnull().sum()
    print(f"    Densidad nula: {nulos_dens} filas")
    if nulos_dens > 0:
        print(f"    → Esos son los municipios con area_km2 = 0 (división por cero).")

# ------------------------------------------------------------
# 11.10 Verificación de población
# ------------------------------------------------------------
# 🇩🇪 La población total de Alemania ronda los 83 millones de personas.
#    Si la suma se aleja mucho de ese número, algo se rompió al
#    convertir los datos. También verificamos que male + female
#    sea aproximadamente igual a population_total.
print(f"\n[11.10] 🇩🇪 VERIFICACIÓN DE POBLACIÓN:")
if "population_total" in df_check.columns:
    print(f"    Población total del país: {df_check['population_total'].sum():,.0f}")
    print(f"    → Alemania tiene ~83 millones de habitantes.")
    print(f"    Población → min: {df_check['population_total'].min():>10.0f} | "
          f"max: {df_check['population_total'].max():>10.0f}")
    if "male" in df_check.columns and "female" in df_check.columns:
        suma_mf = df_check["male"].sum() + df_check["female"].sum()
        diff = abs(suma_mf - df_check["population_total"].sum())
        print(f"    Suma male + female: {suma_mf:,.0f}")
        print(f"    Diferencia vs population_total: {diff:,.0f}")
        print(f"    → Si la diferencia es pequeña, los subtotales cuadran")
        print(f"      con el total. Si es enorme, hay un error de captura.")

# ------------------------------------------------------------
# 11.11 Verificación de estados únicos
# ------------------------------------------------------------
# 🗾 Alemania tiene 16 estados federados (Bundesländer). Si después
#    de limpiar hay 16 estados únicos, todo cuadra. Si hay más, es
#    porque se colaron variantes mal escritas.
print(f"\n[11.11] 🗾 ESTADOS ÚNICOS (state):")
if "state" in df_check.columns:
    estados = df_check["state"].dropna().unique()
    print(f"    Total estados: {len(estados)}")
    print(f"    → Alemania tiene 16 estados federados. Si aquí aparecen más,")
    print(f"      es que hay duplicados con ortografía distinta.")
    for e in sorted(estados):
        print(f"      · {e}")

# ------------------------------------------------------------
# 11.12 Resultado final
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("✅ VALIDACIONES COMPLETADAS")
print("=" * 60)
print(f"📁 Archivo: germany_municipalities_clean.csv")
print(f"📊 Filas:   {df_check.shape[0]:,}")
print(f"📋 Columnas: {df_check.shape[1]}")
print(f"✂️  Eliminadas: {len(cols_eliminadas)} columnas")
print("=" * 60)
print(f"→ Con esto, la tabla de municipios alemanes quedó lista")
print(f"  para el análisis: sin coordenadas rotas, sin textos repetidos,")
print(f"  y con la densidad recalculada en base a datos confiables.")

------------------------------------------------------------
INICIO LIMPIEZA: germany_municipalities
------------------------------------------------------------

[1] 📥 Cargado: 10950 filas x 13 columnas
    → Es como abrir un directorio telefónico de todo un país:
      10,950 municipios (filas), cada uno con 13 datos (columnas).
    Columnas: ['state_(land)', 'district_(kreis)', 'regional_designation', 'municipality_name_(gemeindename)', 'municipality_designation', 'area_km2', 'population_total', 'male', 'female', 'per_km2', 'postal_code', 'longitude', 'latitude']

[2] 🏷️ Rename hecho. Columnas nuevas: ['state', 'district', 'regional_type', 'municipality_name', 'municipality_type', 'area_km2', 'population_total', 'male', 'female', 'population_density', 'postal_code', 'longitude', 'latitude']
    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.
      Más corto, más claro, más fácil de escribir en el código.

-----------------------------------------------------

In [2]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TABLA SEGUROS — Limpieza completa paso a paso
# ------------------------------------------------------------
# Esta parte del código es como la receta de cocina para dejar
# nuestra tabla de seguros lista para el análisis.
# Vamos paso a paso, revisando y arreglando cada detalle,
# con semáforos de colores para que se vea fácil qué está bien
# y qué hay que arreglar.
print("-" * 60)
print("INICIO LIMPIEZA: insurance_policies_claims")
print("-" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
# 📂 Abrimos el CSV como quien abre una caja con papeles viejos:
#    todavía no sabemos qué hay adentro ni cómo está ordenado.
# Pedimos que lea TODO como texto (dtype=str) y que NO convierta
# los espacios vacíos en nulos automáticamente (keep_default_na=False).
# ¿Por qué? Porque primero queremos ver el archivo tal cual está,
# sin que Pandas "adivine" cosas por nosotros. Más vale revisar
# con nuestros propios ojos antes de dejar que el programa decida.
df_in = pd.read_csv(
    "insurance_policies_claims_2018_2025.csv",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_in.shape[0]} filas x {df_in.shape[1]} columnas")
print(f"    → Es como si hubiéramos vaciado la caja sobre la mesa:")
print(f"      {df_in.shape[0]} papeles (filas), cada uno con {df_in.shape[1]} datos (columnas).")

# ------------------------------------------------------------
# 1.5 CARGAR TABLA DE MUNICIPIOS DE ALEMANIA (para el mapeo de CP)
# ------------------------------------------------------------
# 🇩🇪 Esta tabla tiene los códigos postales REALES de Alemania.
#    La vamos a usar como "diccionario" para arreglar los códigos
#    postales ficticios que trae la tabla de seguros.
print("\n[1.5] 🇩🇪 Cargando tabla de municipios de Alemania...")
df_germany = pd.read_csv(
    "germany_municipalities_jun_2025.csv",
    dtype=str,
    keep_default_na=False,
)
print(f"    → Cargada: {df_germany.shape[0]} filas x {df_germany.shape[1]} columnas")
print(f"    → Esta tabla SÍ tiene códigos postales reales alemanes (5 dígitos).")

# ------------------------------------------------------------
# 2. RENAME
# ------------------------------------------------------------
# 🏷️ Aquí revisamos cómo se llaman las columnas. Es como revisar las
#    etiquetas de los cajones antes de empezar a guardar cosas.
# En este caso, ya vienen bien nombradas (snake_case, sin espacios,
# en minúsculas). No hay que hacer nada.
print(f"\n[2] 🏷️ Columnas ya estandarizadas (no se requiere rename)")
print(f"    → Los nombres ya están claros: 'premium_amount_eur', 'policy_id'...")
print(f"      No hay que traducir ni arreglar nada por acá.")

# ------------------------------------------------------------
# 2.1 DROP DE COLUMNAS QUE NO VAMOS A USAR
# ------------------------------------------------------------
# ✂️ Hay columnas que no van a aportar nada al análisis. En este caso,
#    "customer_email" no la vamos a usar ni para agrupar ni para predecir.
#    Es como quitar de la mesa los papeles que no vamos a leer.
cols_a_eliminar = ["customer_email"]
cols_presentes = [c for c in cols_a_eliminar if c in df_in.columns]

if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[2.1] ✂️ Columnas eliminadas: {cols_presentes}")
else:
    print(f"\n[2.1] ✂️ No había columnas que eliminar")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS Y VACÍOS
# ------------------------------------------------------------
# 🩺 Antes de operar, hay que diagnosticar. Esta parte es como ir al
#    doctor y hacerse unos análisis de sangre para ver qué anda mal.
# Contamos cuántos vacíos (cadenas "") y cuántos nulos reales (NaN)
# hay por cada columna, y lo mostramos en una tabla bien ordenada
# de la columna con más problemas a la que menos tiene.
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_in.dtypes,
    "vacios_str":      (df_in == "").sum(),
    "nan_reales":      df_in.isnull().sum(),
    "total_problemas": ((df_in == "") | df_in.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (
    reporte_nulos["total_problemas"] / len(df_in) * 100
).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)

print(reporte_nulos[reporte_nulos["total_problemas"] > 0].to_string())

# 🚦 Y ahora, el semáforo: en vez de leer una tabla llena de números,
#    le ponemos un color a cada columna según qué tan grave es el problema.
#    Es como un tablero de auto: verde = todo bien, rojo = peligro.
print("\nSemáforo por columna:")
for col in reporte_nulos.index:
    pct = reporte_nulos.loc[col, "pct_del_total"]
    if   pct == 0:   icono = "✅"  # Perfecto, sin problemas.
    elif pct < 5:    icono = "🟢"  # Casi limpio, muy poquitos vacíos.
    elif pct < 30:   icono = "🟡"  # Ojo, hay que prestarle atención.
    elif pct < 70:   icono = "🟠"  # Bastante sucio, hay que trabajarlo.
    else:            icono = "🔴"  # Casi vacío, cuidado con esta columna.
    print(f"  {icono}  {col:<35} → {pct:>6.2f}%  "
          f"({reporte_nulos.loc[col, 'total_problemas']} filas)")

# Ahora que ya sabemos cuántos vacíos hay, los convertimos en
# nulos reales (NaN). Es como marcar con resaltador las casillas
# que están en blanco, para que después las podamos encontrar fácil.
df_in = df_in.replace({"": np.nan})
print("\n[2.5] 🔄 Vacíos convertidos a NaN reales.")

# ------------------------------------------------------------
# 2.75 ESTANDARIZAR TEXTO
# ------------------------------------------------------------
# 🔤 Los datos escritos a mano son un caos: "SÍ", "sí", "si ", " SI".
#    Es como si cada persona llenara el mismo formulario a su manera.
#    Para poder agrupar y comparar, hay que dejar todo igualito.
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

# Categorías: las dejamos en minúsculas y les quitamos los espacios
# de los lados. Es como pasar la plancha a la ropa arrugada.
cat_cols = [
    "policy_status", "claim_status", "insurance_type", "coverage_type",
    "payment_frequency", "underwriting_decision",
]
for col in cat_cols:
    df_in[col] = df_in[col].str.lower().str.strip()

print("Estandarizadas (minúsculas + strip):")
for col in cat_cols:
    valores = df_in[col].dropna().unique()[:6]
    print(f"  · {col:<25} → {list(valores)}")
print(f"  → Antes había 'Active', 'ACTIVE', 'active'; ahora todos son 'active'.")

# 🆔 Los IDs y códigos son como las cédulas: sensibles a mayúsculas.
#    "CL-1001" no es lo mismo que "cl-1001". Solo les quitamos
#    los espacios de los lados, sin cambiarles nada más.
id_cols = ["policy_id", "policy_number", "customer_id", "agent_id",
           "broker_id", "adjuster_id", "claim_id", "postal_code"]
for col in id_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.strip()

# 🧑 Para nombres y direcciones, quitamos espacios de las orillas
#    y también los dobles espacios internos. Así "Juan   Pérez"
#    se convierte en "Juan Pérez".
name_cols = ["customer_name", "agent_name", "broker_id",
             "adjuster_name", "address", "city"]
for col in name_cols:
    if col in df_in.columns:
        df_in[col] = (
            df_in[col]
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

# 📞 Teléfonos: vienen con guiones, paréntesis, puntos... un desastre.
#    La idea es dejar solo dígitos (y el '+' si empieza con él).
#    Si después de limpiar quedan menos de 8 dígitos, es porque
#    el número está incompleto, así que lo tratamos como nulo.
def normalizar_telefono(v):
    if pd.isna(v):
        return np.nan
    s = "".join(c for c in str(v).strip() if c.isdigit() or c == "+")
    return s if len(s) >= 8 else np.nan

if "customer_phone" in df_in.columns:
    df_in["customer_phone"] = df_in["customer_phone"].apply(normalizar_telefono)

print("\n📞 Teléfonos normalizados")
print(f"  → '(888) 123-4567' ahora se ve como '+8881234567'.")

# ------------------------------------------------------------
# GÉNERO: M, F, Other y No indica (formato consistente)
# ------------------------------------------------------------
# ⚧️ Igual que con los teléfonos, el género viene escrito de mil maneras:
#     'M', 'Male', 'male', 'other', 'no indica'... Todas esas variantes
#     significan lo mismo pero Pandas las trata como cosas distintas.
#     Las unificamos en 4 categorías estándar.
def normalizar_genero(v):
    if pd.isna(v):
        return np.nan
    g = str(v).strip().lower()
    if g in ("m", "male"):
        return "M"
    if g in ("f", "female"):
        return "F"
    if g in ("other", "o", "divers", "diverse", "non-binary", "nonbinary"):
        return "Other"
    if g in ("no indica", "no indica.", "n/a", "na", "unknown",
             "prefiero no decir", "prefiero no indicar", "-", ""):
        return "No indica"
    # Si el valor no encaja en nada, es mejor dejarlo como nulo
    # antes que inventar una categoría nueva.
    return np.nan

# ⚠️ ¡Ojo! Aquí nos faltaba esta línea. Sin ella, la función se definía
#    pero nunca se aplicaba. Esta es la que hace la magia.
df_in["gender"] = df_in["gender"].apply(normalizar_genero)
print(f"⚧️ Género normalizado: "
      f"{df_in['gender'].value_counts(dropna=False).to_dict()}")
print(f"  → Antes teníamos 'Male', 'm', 'M', 'no indica'... y ahora")
print(f"    solo hay 4 valores posibles: M, F, Other y No indica.")

# ------------------------------------------------------------
# 2.9 FILTRAR SOLO SEGUROS DE VIDA
# ------------------------------------------------------------
# 🎯 Nuestro proyecto se enfoca únicamente en seguros de vida.
#    Es como si de todos los libros de una biblioteca, solo nos
#    interesan los de una sección. Sacamos todos los demás.
print("\n" + "-" * 60)
print("[2.9] 🎯 FILTRO: SOLO SEGUROS DE VIDA")
print("-" * 60)

antes = len(df_in)
print(f"Distribución original de insurance_type:")
print(df_in["insurance_type"].value_counts(dropna=False).to_string())

# Nos quedamos solo con las pólizas de vida. Usamos `.copy()` para
# que Pandas no se queje después si hacemos cambios sobre esta vista.
df_in = df_in[df_in["insurance_type"] == "life"].copy()

print(f"\n📉 Filas eliminadas (no son de vida): {antes - len(df_in)}")
print(f"📊 Filas restantes (solo vida): {len(df_in)}")
print(f"  → Nos quedamos con el {len(df_in)/antes:.1%} de las pólizas originales.")

# ------------------------------------------------------------
# 3. BUGS CRÍTICOS → dropna
# ------------------------------------------------------------
# 🫀 Hay datos que son como el "corazón" de una fila. Si faltan,
#    la fila entera pierde sentido y no podemos confiar en ella.
#    Por ejemplo: ¿de qué sirve una póliza sin ID? ¿O sin prima?
#    ¿O sin fecha de inicio? Nada. Las eliminamos sin piedad.
antes = len(df_in)
df_in = df_in.dropna(subset=[
    "policy_id",           # 🔑 Llave primaria: sin esto, no sabemos a qué se refiere.
    "customer_id",         # 👤 Dueño de la póliza: el cliente es fundamental.
    "premium_amount_eur",  # 💰 Prima cobrada: dato clave para cualquier auditoría.
    "policy_start_date",   # 📅 Sin fecha, no sabemos cuánto tiempo lleva activa.
])
print(f"\n[3] 🫀 Filas eliminadas por falta de ID/prima/fecha: {antes - len(df_in)}")
print(f"    ✅ Filas restantes: {len(df_in)}")
print(f"  → Es como tirar cartas rotas: si les falta el palo o el número,")
print(f"    no sirven para jugar, aunque el resto esté en buen estado.")

# ------------------------------------------------------------
# 4. NULOS LÓGICOS
# ------------------------------------------------------------
# 🤔 Aquí hay un matiz importante: no todos los nulos son errores.
#    A veces un nulo significa "esto no aplica" o "esto no ocurrió".
#    Por ejemplo, si un cliente nunca puso una queja, la columna
#    'complaint_type' está vacía NO porque falte el dato, sino porque
#    no hay queja que registrar. Rellenamos con etiquetas lógicas.
df_in["customer_phone"]          = df_in["customer_phone"].fillna("Sin Teléfono")
df_in["claim_denial_reason"]     = df_in["claim_denial_reason"].fillna("No Aplica")
df_in["complaint_type"]          = df_in["complaint_type"].fillna("Sin Queja")
df_in["police_report_number"]    = df_in["police_report_number"].fillna("Sin Reporte Policial")
df_in["adjuster_id"]             = df_in["adjuster_id"].fillna("Sin Asignar")
df_in["adjuster_name"]           = df_in["adjuster_name"].fillna("Sin Asignar")
df_in["outstanding_balance_eur"] = df_in["outstanding_balance_eur"].fillna(0)
df_in["fraud_flag"]              = df_in["fraud_flag"].fillna(False).astype(bool)

print(f"\n[4] 🤔 Nulos lógicos rellenados (teléfono, denial, complaint, etc.)")
print(f"  → En vez de dejar casillas en blanco sin explicación, ahora")
print(f"    dicen claramente 'Sin Queja', 'No Aplica', 'Sin Asignar'...")

# ------------------------------------------------------------
# 5. COLUMNAS DE VEHÍCULO Y PROPIEDAD
# ------------------------------------------------------------
# 🚗🏠 Como filtramos solo seguros de vida, estas columnas quedaron
#   100% vacías. No tiene sentido rellenarlas una por una con
#   "No Aplica"; es como guardar cajas vacías en la bodega
#   solo por si algún día las usamos. Mejor las tiramos.
cols_no_aplican_vida = [
    "vehicle_make", "vehicle_model", "vehicle_year", "vehicle_value_eur",
    "property_type", "property_value_eur", "property_year_built",
]
cols_presentes = [c for c in cols_no_aplican_vida if c in df_in.columns]

# 🩺 Diagnóstico: primero confirmamos que efectivamente están vacías.
for col in cols_presentes:
    nulos = df_in[col].isna().sum()
    pct = nulos / len(df_in) * 100 if len(df_in) > 0 else 0
    icono = "🔴" if pct > 70 else "🟠"
    print(f"    {icono} {col:<25} → {pct:>6.2f}% nulos")

# ✂️ Y las eliminamos.
if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[5] ✂️ Columnas eliminadas (no aplican a Vida): {cols_presentes}")
    print(f"  → Como filtramos solo vida, estas columnas eran como cajones")
    print(f"    vacíos: existían pero no aportaban nada.")

# ------------------------------------------------------------
# 6. CLAIM: nulos dependen de si hay siniestro
# ------------------------------------------------------------
# 💡 Aquí hay una lógica distinta: los nulos en las columnas de reclamo
#    dependen de si hubo o no un siniestro. Si NO hubo siniestro, es
#    totalmente lógico que no haya monto de reclamo: ese nulo ES la
#    respuesta correcta (cero euros). No es un error.
sin_claim = df_in["claim_id"].isna()
print(f"\n[6] 💡 Pólizas SIN siniestro: {sin_claim.sum()}")
print(f"    Pólizas CON siniestro:   {(~sin_claim).sum()}")
print(f"  → De cada 100 pólizas, más o menos {sin_claim.mean()*100:.0f} nunca tuvieron")
print(f"    un reclamo. Para esas, el monto correcto es 0, no un nulo.")

# Rellenamos con 0 para las pólizas sin siniestro.
df_in.loc[sin_claim, "claim_amount_eur"] = (
    df_in.loc[sin_claim, "claim_amount_eur"].fillna(0)
)
df_in.loc[sin_claim, "claim_approved_amount_eur"] = (
    df_in.loc[sin_claim, "claim_approved_amount_eur"].fillna(0)
)
df_in.loc[sin_claim, "settlement_amount_eur"] = (
    df_in.loc[sin_claim, "settlement_amount_eur"].fillna(0)
)

# ⚠️ Pero al revés: si una póliza SÍ tiene siniestro pero le falta el
#    monto de liquidación, eso sí es un bug crítico. Algo se rompió
#    en el sistema, no podemos confiar en esa fila. La eliminamos.
antes = len(df_in)
mask_bug = (~sin_claim) & (df_in["settlement_amount_eur"].isna())
print(f"    🫀 Filas eliminadas (siniestro sin settlement): {mask_bug.sum()}")
print(f"  → Son como facturas sin total: si hay reclamo pero no hay monto,")
print(f"    no se puede auditar nada.")
df_in = df_in[~mask_bug]

# ------------------------------------------------------------
# 7. EDAD: mediana agrupada por género
# ------------------------------------------------------------
# 📊 Para rellenar edades faltantes, hay una decisión importante:
#    NO usamos un promedio general. ¿Por qué? Porque hombres y mujeres
#    suelen tener distribuciones de edad distintas en este tipo de
#    seguros, así que imputar con la mediana de cada grupo es mucho
#    más preciso que meter un solo número para todos.
df_in["age"] = pd.to_numeric(df_in["age"], errors="coerce")
antes_nulos = df_in["age"].isna().sum()
print(f"\n[7] 📊 Edades nulas antes de imputar: {antes_nulos}")

mediana_por_genero = df_in.groupby("gender", dropna=False)["age"].median()
print(f"    Mediana de edad por género: {mediana_por_genero.to_dict()}")
print(f"  → Es como rellenar los huecos con el valor más típico del grupo")
print(f"    al que pertenece cada fila, no con el típico de todo el dataset.")

# Rellenamos cada nulo con la mediana de su propio grupo.
df_in["age"] = df_in.groupby("gender", dropna=False)["age"].transform(
    lambda s: s.fillna(s.median())
)
# Si algún grupo quedó sin mediana (por ejemplo, todos sus valores
# eran nulos), usamos la mediana global como último recurso.
df_in["age"] = df_in["age"].fillna(df_in["age"].median())
print(f"    ✅ Edades nulas después: {df_in['age'].isna().sum()}")

# ------------------------------------------------------------
# 7.5 MAPEO DE CÓDIGOS POSTALES FICTICIOS → REALES
# ------------------------------------------------------------
# 📮 Aquí viene una de las partes más importantes: los códigos postales
#    de la tabla de seguros son FICTICIOS (no existen en Alemania).
#    Es como si tuvieras direcciones inventadas: "Calle Falsa 123".
#
#    Para poder hacer un merge con la tabla de Alemania (que SÍ tiene
#    códigos postales reales de 5 dígitos), necesitamos reemplazar
#    esos códigos ficticios por códigos reales.
#
#    ¿Cómo? Usando la CIUDAD y el ESTADO de cada póliza como "pista"
#    para buscar el código postal real en la tabla de Alemania.
#
#    Es como si alguien te da una dirección con el número de casa mal,
#    pero te dice "vivo en Berlín": con eso puedes buscar el código
#    postal correcto de Berlín en una guía telefónica.
print("\n" + "-" * 60)
print("[7.5] 📮 MAPEO DE CÓDIGOS POSTALES FICTICIOS → REALES")
print("-" * 60)

def mapear_codigos_postales_reales(
    df_seguros: pd.DataFrame,
    df_germany: pd.DataFrame,
    col_city_seguros: str = "city",
    col_state_seguros: str = "state",
    col_city_germany: str = "municipality_name_(gemeindename)",
    col_state_germany: str = "state_(land)",
    col_postal_germany: str = "postal_code",
) -> pd.DataFrame:
    """
    Reemplaza los códigos postales ficticios de la tabla de seguros
    por códigos postales reales alemanes, haciendo match por ciudad/estado
    contra la tabla de municipios de Alemania.

    Estrategia de match (en orden de prioridad):
      1. Match exacto por (estado, ciudad)  → más preciso
      2. Match exacto solo por ciudad       → buena precisión
      3. Match parcial (substring)          → último recurso
    """

    df = df_seguros.copy()

    # 1) Guardar el código original por si necesitamos trazabilidad
    df["postal_code_original"] = df["postal_code"]

    # 2) Función de normalización de texto (minúsculas, sin tildes,
    #    sin espacios extra). Así "München" y "munchen" son iguales.
    def _norm(s):
        if pd.isna(s):
            return np.nan
        s = str(s).strip().lower()
        # Quitar diéresis y eszett para normalizar
        reemplazos = {"ü": "u", "ö": "o", "ä": "a", "ß": "ss",
                      "é": "e", "è": "e", "á": "a", "à": "a"}
        for k, v in reemplazos.items():
            s = s.replace(k, v)
        return s

    # 3) Preparar tabla de Alemania con textos normalizados
    df_ger = df_germany.copy()
    df_ger["_city_norm"]  = df_ger[col_city_germany].apply(_norm)
    df_ger["_state_norm"] = df_ger[col_state_germany].apply(_norm)
    df_ger["_cp"] = df_ger[col_postal_germany].astype(str).str.strip()

    # 4) Diccionario 1: (estado, ciudad) → código postal
    #    Es el más preciso porque evita ambigüedades.
    df_ger_unicos = df_ger.dropna(subset=["_state_norm", "_city_norm", "_cp"])
    df_ger_unicos = df_ger_unicos.drop_duplicates(subset=["_state_norm", "_city_norm"])
    mapa_estado_ciudad = (
        df_ger_unicos
        .set_index(["_state_norm", "_city_norm"])["_cp"]
        .to_dict()
    )

    # 5) Diccionario 2: ciudad sola → código postal
    #    Útil cuando el estado no coincide (por ejemplo, "Bavaria" vs "Bayern").
    df_ger_solo_ciudad = df_ger.dropna(subset=["_city_norm", "_cp"])
    df_ger_solo_ciudad = df_ger_solo_ciudad.drop_duplicates(subset=["_city_norm"])
    mapa_ciudad = (
        df_ger_solo_ciudad
        .set_index("_city_norm")["_cp"]
        .to_dict()
    )

    # 6) Lista de ciudades únicas de Alemania para match parcial
    ciudades_alemania = list(mapa_ciudad.keys())

    # 7) Función que decide el código postal para una fila
    def _asignar_cp(row):
        ciudad = _norm(row.get(col_city_seguros))
        estado = _norm(row.get(col_state_seguros))

        if pd.isna(ciudad):
            return np.nan

        # Intento 1: match exacto por (estado, ciudad)
        if (estado, ciudad) in mapa_estado_ciudad:
            return mapa_estado_ciudad[(estado, ciudad)]

        # Intento 2: match exacto solo por ciudad
        if ciudad in mapa_ciudad:
            return mapa_ciudad[ciudad]

        # Intento 3: match parcial (substring)
        # Ej: "Hanover" debería encontrar "Hannover" o "Hanover"
        for ciudad_ger in ciudades_alemania:
            if ciudad in ciudad_ger or ciudad_ger in ciudad:
                return mapa_ciudad[ciudad_ger]

        # Intento 4: casos especiales (nombres en inglés)
        alias = {
            "munich": "munchen",
            "cologne": "koln",
            "hanover": "hannover",
            "nuremberg": "nurnberg",
            "vienna": "wien",
            "brunswick": "braunschweig",
            "mayence": "mainz",
        }
        ciudad_alias = alias.get(ciudad, ciudad)
        if ciudad_alias in mapa_ciudad:
            return mapa_ciudad[ciudad_alias]

        # Si nada funcionó, NaN
        return np.nan

    # 8) Aplicar el mapeo a cada fila
    df["postal_code"] = df.apply(_asignar_cp, axis=1)

    # 9) Reporte de resultados
    total = len(df)
    mapeados = df["postal_code"].notna().sum()
    print(f"    📊 Total de filas: {total}")
    print(f"    ✅ Mapeados: {mapeados} ({mapeados/total:.1%})")
    print(f"    ❌ Sin mapear: {total - mapeados} ({(total-mapeados)/total:.1%})")

    # 10) Mostrar las ciudades que no se pudieron mapear
    sin_match = df[df["postal_code"].isna()][col_city_seguros].dropna().unique()
    if len(sin_match) > 0:
        print(f"\n    ⚠️ Ciudades sin match ({len(sin_match)}):")
        for c in list(sin_match)[:20]:
            print(f"       · {c}")

    return df

# Aplicamos la función a nuestra tabla de seguros
df_in = mapear_codigos_postales_reales(
    df_seguros=df_in,
    df_germany=df_germany,
)

print(f"\n[7.5] ✅ Códigos postales mapeados a formato alemán real")
print(f"    → Ejemplo de antes vs después:")
ejemplo = df_in[["city", "postal_code_original", "postal_code"]].head(5)
print(ejemplo.to_string(index=False))

# ------------------------------------------------------------
# 8. COLUMNAS CON NULOS RESTANTES (RESUMEN)
# ------------------------------------------------------------
# 🔍 Hacemos una última pasada para ver qué nulos quedaron vivos.
#    Algunos son a propósito (por ejemplo, fechas de siniestro
#    todavía abiertos), pero queremos verlo con claridad.
print("\n" + "-" * 60)
print("[8] 🔍 NULOS RESTANTES POR COLUMNA (post-limpieza)")
print("-" * 60)
nulos_finales = df_in.isnull().sum().sort_values(ascending=False)
nulos_finales = nulos_finales[nulos_finales > 0]
if len(nulos_finales) == 0:
    print("  ✅ Sin nulos restantes")
    print("  → La tabla quedó limpia como un vidrio recién lavado.")
else:
    print(nulos_finales.to_string())
    print("\n  ⚠️  Estas columnas conservan nulos a propósito:")
    print("     - Fechas de siniestro abierto (NaT es correcto)")
    print("     - Género con valor irreconocible (texto basura)")
    print("     - Códigos postales que no se pudieron mapear")

# ------------------------------------------------------------
# 9. AUDITORÍA FINAL
# ------------------------------------------------------------
# 🧾 Chequeo final antes de cerrar la caja. Es como repasar la lista
#    del supermercado antes de salir de la tienda: nos aseguramos
#    de que todo esté y que nada se nos haya escapado.
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)
print(f"Filas finales: {len(df_in)}")
print(f"Columnas finales: {df_in.shape[1]}")
print(f"\n🔑 Duplicados en policy_id: {df_in.duplicated(subset=['policy_id']).sum()}")
print(f"  → Si es 0, cada póliza es única. Si es >0, hay que investigar.")
print(f"\n⚧️ Distribución de género (solo Vida):")
print(df_in["gender"].value_counts(dropna=False).to_string())
print(f"\n🎯 Distribución de insurance_type (debe ser solo 'life'):")
print(df_in["insurance_type"].value_counts(dropna=False).to_string())
print(f"\n👀 Primeras 3 filas:")
print(df_in.head(3).to_string())
print(f"\n📋 Tipos de datos:")
print(df_in.dtypes.head(15))

# ------------------------------------------------------------
# 9.5 MERGE CON TABLA DE ALEMANIA
# ------------------------------------------------------------
# 🔗 Ahora que ya tenemos códigos postales reales, podemos hacer
#    el merge con la tabla de municipios de Alemania. Es como
#    unir dos piezas de un rompecabezas que antes no encajaban.
print("\n" + "-" * 60)
print("[9.5] 🔗 MERGE CON TABLA DE ALEMANIA")
print("-" * 60)

antes_merge = len(df_in)
df_merged = df_in.merge(
    df_germany,
    left_on="postal_code",
    right_on="postal_code",
    how="left",
    suffixes=("_seguro", "_germany"),
)

con_match = df_merged["municipality_name_(gemeindename)"].notna().sum()
print(f"    📊 Filas antes del merge: {antes_merge}")
print(f"    📊 Filas después del merge: {len(df_merged)}")
print(f"    ✅ Filas con match en Alemania: {con_match} ({con_match/antes_merge:.1%})")
print(f"    ❌ Filas sin match: {antes_merge - con_match}")

if con_match > 0:
    print(f"\n    👀 Ejemplo de filas con match:")
    cols_ejemplo = ["policy_id", "city_seguro", "postal_code",
                    "municipality_name_(gemeindename)", "state_(land)"]
    cols_ejemplo = [c for c in cols_ejemplo if c in df_merged.columns]
    print(df_merged[cols_ejemplo].dropna().head(5).to_string(index=False))

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos la tabla limpia en un CSV nuevo. Es como sacar una
#    foto del trabajo terminado, para no tener que repetir todo
#    la próxima vez que abramos el archivo.
df_in.to_csv("insurance_life_only_clean.csv", index=False)
print(f"\n✅ Guardado: insurance_life_only_clean.csv ({len(df_in)} filas)")

# 💾 También guardamos la tabla con el merge aplicado
df_merged.to_csv("insurance_life_merged_germany.csv", index=False)
print(f"✅ Guardado: insurance_life_merged_germany.csv ({len(df_merged)} filas)")

# 📸 Verificación post-guardado: volvemos a abrir el archivo que
#    acabamos de escribir, para confirmar que quedó como pensábamos.
#    Es como revisar la foto que acabamos de sacar.
df_check = pd.read_csv("insurance_life_only_clean.csv")
print("\n--- 🔍 VERIFICACIÓN POST-GUARDADO ---")
print("⚧️ Género:", df_check["gender"].value_counts(dropna=False).to_dict())
print("🎯 Insurance type:", df_check["insurance_type"].value_counts(dropna=False).to_dict())
print("🚗 ¿Quedan columnas de vehículo?:",
      [c for c in df_check.columns if "vehicle" in c or "property" in c])
print("📧 ¿Queda customer_email?:", "customer_email" in df_check.columns)
print("📮 ¿Queda postal_code_original?:", "postal_code_original" in df_check.columns)

# ------------------------------------------------------------
# 11. LISTADO DE COLUMNAS FINALES
# ------------------------------------------------------------
# 📜 Imprimimos la lista completa de columnas que sobrevivieron a la
#    limpieza. Es como el inventario de lo que quedó en la bodega
#    después de tirar todo lo que no servía.
print("\n" + "-" * 60)
print("📜 COLUMNAS FINALES DEL DATASET")
print("-" * 60)
for i, col in enumerate(df_in.columns, 1):
    print(f"  {i:>2}. {col}")

print(f"\n📊 Total: {df_in.shape[1]} columnas")
print("\n" + "=" * 60)
print("🎉 LIMPIEZA COMPLETADA")
print("=" * 60)

------------------------------------------------------------
INICIO LIMPIEZA: insurance_policies_claims
------------------------------------------------------------

[1] 📥 Cargado: 52676 filas x 76 columnas
    → Es como si hubiéramos vaciado la caja sobre la mesa:
      52676 papeles (filas), cada uno con 76 datos (columnas).

[1.5] 🇩🇪 Cargando tabla de municipios de Alemania...
    → Cargada: 10950 filas x 13 columnas
    → Esta tabla SÍ tiene códigos postales reales alemanes (5 dígitos).

[2] 🏷️ Columnas ya estandarizadas (no se requiere rename)
    → Los nombres ya están claros: 'premium_amount_eur', 'policy_id'...
      No hay que traducir ni arreglar nada por acá.

[2.1] ✂️ Columnas eliminadas: ['customer_email']

------------------------------------------------------------
[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS
------------------------------------------------------------
                            tipo_dato  vacios_str  nan_reales  total_problemas  pct_del_total
property_type   

In [3]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# ENRIQUECIMIENTO — Cruzar Seguros con Municipios de Alemania
# ------------------------------------------------------------
# 🔗 Este script une dos tablas que ya limpiamos antes:
#    1. Seguros de vida (insurance_life_only_clean.csv)
#    2. Municipios de Alemania (germany_municipalities_clean.csv)
#    La idea es enriquecer la tabla de seguros con datos geográficos:
#    cuánta población tiene el municipio del cliente, qué tan denso
#    es, en qué estado federado vive, sus coordenadas, etc.
#    Es como tomar dos rompecabezas y encontrar las piezas que
#    encajan por el código postal para armar uno más grande.

# ------------------------------------------------------------
# FUNCIÓN GLOBAL DE ESTANDARIZACIÓN
# ------------------------------------------------------------
# 🏷️ Antes de cruzarlas, necesitamos que las columnas de ambas
#    tablas se llamen igual. Esta función deja los nombres en
#    snake_case: minúsculas, sin puntuación, sin espacios, con
#    guiones bajos en lugar de espacios.
#    Es como ponerle la misma etiqueta a todos los cajones del
#    archivero, sin importar quién los etiquetó originalmente.
def estandarizar_nombres(df):
    """snake_case + minúsculas + sin puntuación + sin espacios."""
    df.columns = (
        df.columns
        .str.strip()                             # 🧹 Sin espacios en las orillas
        .str.lower()                             # 🔡 Todo en minúsculas
        .str.replace(r"[^\w\s]", "", regex=True) # ✂️ Fuera puntuación rara
        .str.replace(r"\s+", "_", regex=True)    # 🔗 Espacios → guion bajo
        .str.replace(r"_+", "_", regex=True)     # 🧽 Sin guiones dobles
        .str.strip("_")                          # 🧼 Sin guion al inicio/final
    )
    return df

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
# 📂 Abrimos las dos tablas limpias que guardamos en pasos previos.
#    Igual que antes: todo como texto y sin dejar que Pandas invente
#    nulos por su cuenta. Primero ver qué hay, después decidir.
df_seguros = pd.read_csv("insurance_life_only_clean.csv", dtype=str, keep_default_na=False)
df_alemania = pd.read_csv("germany_municipalities_clean.csv", dtype=str, keep_default_na=False)

# ------------------------------------------------------------
# 2. ESTANDARIZAR NOMBRES BASE (snake_case en ambas)
# ------------------------------------------------------------
# 🏷️ Aplicamos la función de arriba a las dos tablas. Así nos
#    aseguramos de que las columnas se llamen igual si representan
#    lo mismo (por ejemplo: "postal_code" en ambas, no una con
#    "Postal_Code" y otra con "postal-code").
df_seguros = estandarizar_nombres(df_seguros)
df_alemania = estandarizar_nombres(df_alemania)

print("📋 Columnas Seguros (estandarizadas):")
print(df_seguros.columns.tolist())
print("\n📋 Columnas Alemania (estandarizadas):")
print(df_alemania.columns.tolist())
print(f"\n  → Nombres en minúsculas, sin espacios, listos para cruzarlos.")

# ------------------------------------------------------------
# 3. NORMALIZAR LA LLAVE EN AMBAS TABLAS
# ------------------------------------------------------------
# 🔑 La "llave" que une las dos tablas es el código postal. Pero
#    viene escrito de mil formas: "10115", "10115.0" (por culpa
#    de Excel), "10-115" (con guion), " 10115 " (con espacios)...
#    Es como buscar el mismo número de teléfono cuando alguien lo
#    escribió con paréntesis, con guiones, o sin nada.
#    Esta función lo deja siempre como 5 dígitos limpios.
def normalizar_cp(serie):
    return (
        serie.astype(str)
        .str.strip()                              # 🧹 Quitar espacios
        .str.replace(r"\.0$", "", regex=True)     # 🧹 Quitar el ".0" que mete Excel
        .str.replace(r"[^\d]", "", regex=True)    # 🔢 Solo dejar dígitos
        .replace({"": np.nan, "nan": np.nan})     # 🕳️ Vacíos → nulos reales
        .str.zfill(5)                             # 🧩 Rellenar con ceros a la izquierda
    )

df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

print(f"\n🔑 Llave normalizada en ambas tablas.")
print(f"  → '10115.0', ' 10115 ', '101-15' ahora todos son '10115'.")
print(f"    Sin esto, el merge fallaría porque '10115' != '10115.0'.")

# ------------------------------------------------------------
# 4. RENOMBRAR SEGUROS (prefijo customer_ para lo geográfico)
# ------------------------------------------------------------
# 🏷️ Las columnas geográficas de la tabla de seguros (ciudad,
#    dirección, estado, país) describen al CLIENTE. Le ponemos
#    el prefijo "customer_" para que quede claro que ese dato
#    viene del cliente, no del municipio.
#    Es como ponerle etiqueta de "de Juan" a las cosas que son
#    de Juan, para que no se confundan con las de María.
rename_seguros = {
    "city":    "customer_city",
    "address": "customer_address",
    "state":   "customer_state",
    "country": "customer_country",
}
# Solo renombramos las que efectivamente existen (por si alguna falta).
rename_seguros = {k: v for k, v in rename_seguros.items()
                  if k in df_seguros.columns}
df_seguros = df_seguros.rename(columns=rename_seguros)

print(f"\n🏷️ Seguros renombradas: {rename_seguros}")
print(f"  → Ahora sabemos que 'customer_city' es la ciudad del cliente,")
print(f"    no la del municipio alemán. Sin ambigüedades.")

# ------------------------------------------------------------
# 5. RENOMBRAR ALEMANIA (prefijo municipality_)
# ------------------------------------------------------------
# 🏷️ Igual del otro lado: las columnas de la tabla de Alemania
#    describen al MUNICIPIO. Les ponemos el prefijo "municipality_".
#    Así, cuando crucemos las tablas, no habrá confusión entre
#    "customer_state" (estado donde vive el cliente según su dirección)
#    y "municipality_state" (estado federado del municipio según el
#    catálogo oficial). ¡Son cosas distintas aunque suenen parecidas!
rename_alemania = {
    "state":              "municipality_state",
    "district":           "municipality_district",
    "regional_type":      "municipality_regional_type",
    "population_total":   "municipality_population",
    "population_density": "municipality_density",
    "male":               "municipality_male",
    "female":             "municipality_female",
    "latitude":           "municipality_latitude",
    "longitude":          "municipality_longitude",
}
rename_alemania = {k: v for k, v in rename_alemania.items()
                   if k in df_alemania.columns}
df_alemania = df_alemania.rename(columns=rename_alemania)

print(f"🏷️ Alemania renombradas: {rename_alemania}")
print(f"  → Cada columna ahora tiene su 'dueño' claro: customer_ o municipality_.")

# ------------------------------------------------------------
# 6. VERIFICAR LLAVE Y COLISIONES
# ------------------------------------------------------------
# 🚦 Chequeo pre-merge: antes de unir, hay que asegurarse de que
#    todo esté en orden. Es como revisar los frenos antes de un
#    viaje largo por carretera.
print("\n" + "=" * 60)
print("🚦 VERIFICACIÓN PRE-MERGE")
print("=" * 60)

# ✅ Primero: que la llave exista en AMBAS tablas.
assert "postal_code" in df_seguros.columns, "❌ Falta postal_code en Seguros"
assert "postal_code" in df_alemania.columns, "❌ Falta postal_code en Alemania"
print("✅ postal_code existe en ambas tablas")

# ✅ Segundo: que la única columna en común sea la llave. Si hay
#    otras, Pandas les pondría sufijos "_x" y "_y" y ensuciaría
#    el merge. Con esta revisión nos aseguramos de que el merge
#    va a quedar perfectamente limpio.
comunes = set(df_seguros.columns) & set(df_alemania.columns)
print(f"\nColumnas en común: {comunes}")
if comunes == {"postal_code"}:
    print("✅ Solo la llave coincide (merge limpio, sin sufijos)")
    print("  → Cuando unamos, no habrá columnas duplicadas como")
    print("    'state_x' y 'state_y'. Todo quedará claro.")
else:
    print(f"⚠️  Hay {len(comunes) - 1} colisiones además de la llave")
    print(f"  → Podría generar sufijos feos y confusos en el resultado.")

# ------------------------------------------------------------
# 7. CONVERTIR NUMÉRICOS EN ALEMANIA
# ------------------------------------------------------------
# 🔢 Al leer todo como texto, las columnas numéricas de Alemania
#    quedaron como strings. Para hacer sumas, promedios y otros
#    cálculos, hay que convertirlas a número.
#    También limpiamos comas decimales europeas ("13,5" → "13.5")
#    y vacíos que quedaron como texto.
cols_numericas = [
    "municipality_population", "municipality_density",
    "municipality_male", "municipality_female",
    "municipality_latitude", "municipality_longitude",
]
cols_numericas = [c for c in cols_numericas if c in df_alemania.columns]

for col in cols_numericas:
    df_alemania[col] = (
        df_alemania[col]
        .astype(str)
        .str.strip()                                 # 🧹 Sin espacios
        .str.replace(",", ".", regex=False)          # 🔄 Coma → punto decimal
        .replace({"": None, "nan": None, "None": None})  # 🕳️ Textos vacíos → None
    )
    # Convertimos a número. Si algo no se puede, queda como nulo (coerce).
    df_alemania[col] = pd.to_numeric(df_alemania[col], errors="coerce")

print(f"\n🔢 Columnas numéricas convertidas en Alemania:")
for c in cols_numericas:
    print(f"    · {c}")
print(f"  → Antes eran strings, ahora son números listos para sumar y promediar.")

# ------------------------------------------------------------
# 8. AGREGAR ALEMANIA: 1 FILA POR CP
# ------------------------------------------------------------
# 🧮 Un código postal puede tener varios municipios (o un municipio
#    puede aparecer en varias filas). Eso significa que si unimos
#    directamente, cada póliza de seguros se duplicaría.
#    Para evitarlo, agrupamos Alemania a UNA SOLA FILA por código
#    postal. Es como pasar de un listado desordenado de vecinos a
#    un resumen de una sola línea por casa.
#
# Cada tipo de columna se agrupa distinto:
#   · Textos (nombre, estado, distrito): "first" → tomamos el primero.
#     Es como si varios vecinos dijeron el nombre de la calle:
#     con uno basta.
#   · Conteos (población, hombre, mujeres): "sum" → los sumamos.
#     Si un CP tiene 3 municipios, sumamos sus poblaciones.
#   · Promedios (densidad, lat, lon): "mean" → promediamos.
#     La coordenada del CP es el centro aproximado.
agg_dict = {}
for col in ["municipality_name", "municipality_state",
            "municipality_district", "municipality_regional_type"]:
    if col in df_alemania.columns:
        agg_dict[col] = "first"
for col in ["municipality_population", "municipality_male", "municipality_female"]:
    if col in df_alemania.columns:
        agg_dict[col] = "sum"
for col in ["municipality_density", "municipality_latitude", "municipality_longitude"]:
    if col in df_alemania.columns:
        agg_dict[col] = "mean"

df_alemania_agg = df_alemania.groupby("postal_code", as_index=False).agg(agg_dict)
print(f"\n🧮 Alemania agregada: {len(df_alemania)} → {len(df_alemania_agg)} filas")
print(f"  → Se redujo a una sola fila por código postal único.")
print(f"    Ahora sí podemos unir sin duplicar pólizas de seguros.")

# ------------------------------------------------------------
# 9. MERGE LIMPIO (sin suffixes porque ya no hay colisiones)
# ------------------------------------------------------------
# 🔗 ¡El momento clave! Unimos las dos tablas por el código postal.
#    - how="left": conservamos TODAS las pólizas de seguros, aunque
#      el código postal no exista en Alemania (quedaría como nulo
#      en las columnas del municipio).
#    - validate="many_to_one": verificamos que cada código postal
#      en Alemania sea único. Si no lo fuera, Pandas nos avisaría.
#    - indicator=True: nos agrega una columna "_merge" que dice si
#      cada póliza encontró su municipio o no. ¡Perfecto para auditar!
df_final = pd.merge(
    df_seguros,
    df_alemania_agg,
    on="postal_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# 🚦 Semáforo del match: cuántas pólizas encontraron su municipio
#    y cuántas quedaron sin match. Es como ver el marcador del partido.
print("\n🚦 Auditoría del match:")
print(df_final["_merge"].value_counts().to_string())

# Traducimos el resultado del match a un semáforo legible.
match_counts = df_final["_merge"].value_counts()
total = len(df_final)

if "both" in match_counts:
    pct_match = match_counts["both"] / total * 100
    if   pct_match >= 95: icono = "✅"  # Casi todo cruzó, excelente.
    elif pct_match >= 80: icono = "🟢"  # Muy bien, apenas unos huecos.
    elif pct_match >= 50: icono = "🟡"  # Aceptable, hay que revisar.
    else:                 icono = "🔴"  # Mal: la mayoría no cruzó.
    print(f"\n  {icono}  Match exitoso: {match_counts['both']:,} filas ({pct_match:.2f}%)")

if "left_only" in match_counts:
    pct_sin = match_counts["left_only"] / total * 100
    print(f"  ⚠️  Sin match en Alemania: {match_counts['left_only']:,} filas ({pct_sin:.2f}%)")
    print(f"     → Son pólizas cuyo código postal no está en el catálogo alemán.")
    print(f"       Puede ser un CP mal escrito, o de otro país.")

# Ya no necesitamos la columna de auditoría, así que la quitamos.
df_final = df_final.drop(columns=["_merge"])

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos la tabla enriquecida. Es el resultado de todo el
#    trabajo: seguros de vida + datos geográficos del municipio
#    de cada cliente, unidos por el código postal.
df_final.to_csv("insurance_life_enriched.csv", index=False)
print(f"\n✅ Guardado: insurance_life_enriched.csv ({len(df_final):,} filas)")
print(f"   Columnas: {df_final.shape[1]}")
print(f"  → Ahora tenemos una tabla enriquecida: cada póliza viene")
print(f"    con la información del municipio donde vive ese cliente.")
print(f"    Lista para análisis, dashboards o modelos de ML.")

📋 Columnas Seguros (estandarizadas):
['policy_id', 'policy_number', 'customer_id', 'customer_name', 'customer_phone', 'date_of_birth', 'age', 'gender', 'insurance_type', 'policy_status', 'coverage_type', 'policy_start_date', 'policy_end_date', 'renewal_date', 'payment_frequency', 'premium_amount_eur', 'deductible_eur', 'coverage_limit_eur', 'total_paid_eur', 'outstanding_balance_eur', 'address', 'city', 'postal_code', 'state', 'country', 'claim_id', 'claim_date', 'claim_status', 'claim_amount_eur', 'claim_approved_amount_eur', 'claim_description', 'incident_date', 'incident_location', 'police_report_filed', 'police_report_number', 'adjuster_id', 'adjuster_name', 'adjustment_date', 'settlement_date', 'settlement_amount_eur', 'claim_denial_reason', 'fraud_flag', 'fraud_score', 'risk_score', 'underwriting_decision', 'commission_amount_eur', 'agent_id', 'agent_name', 'agent_commission_pct', 'broker_id', 'broker_commission_pct', 'discount_applied_pct', 'no_claims_bonus_pct', 'loyalty_discou

In [5]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# ENRIQUECIMIENTO v2 — Seguros + Municipios + Estados de Alemania
# ------------------------------------------------------------
# 🔗 Este script hace lo mismo que el anterior pero con esteroides:
#    1. Cruza los seguros con el catálogo de municipios por CP.
#    2. Además, cruza por ESTADO para los casos en que el CP no
#       exista en el catálogo alemán. Así rescatamos más información.
#    3. Luego agrupa todo por póliza (policy_id), porque una póliza
#       puede tener varios reclamos asociados y queremos una sola
#       fila por póliza.
#
# Es como si en vez de buscar solo por "código postal exacto",
# también dejáramos abierta la opción de "bueno, si no encuentro
# el municipio exacto, al menos sé en qué estado vive el cliente".
# Y luego, como cada póliza puede tener varios reclamos, los
# colapsamos en una sola fila con el total de reclamos.

# ------------------------------------------------------------
# CARGA
# ------------------------------------------------------------
# 📂 Abrimos las dos tablas limpias de pasos anteriores.
#    Igual que siempre: todo como texto y sin que Pandas invente
#    nulos por su cuenta, para ver el archivo tal cual está.
df_seguros = pd.read_csv("insurance_life_only_clean.csv", dtype=str, keep_default_na=False)
df_alemania = pd.read_csv("germany_municipalities_clean.csv", dtype=str, keep_default_na=False)

print(f"📥 Cargado:")
print(f"    Seguros:  {df_seguros.shape[0]:,} filas x {df_seguros.shape[1]} columnas")
print(f"    Alemania: {df_alemania.shape[0]:,} filas x {df_alemania.shape[1]} columnas")

# ------------------------------------------------------------
# 1. NORMALIZAR CP
# ------------------------------------------------------------
# 🔑 El código postal es la llave principal para cruzar. Pero
#    viene escrito de mil formas: "10115", "10115.0" (por Excel),
#    "10-115", " 10115 "... Es como buscar un mismo número de
#    teléfono cuando cada persona lo escribió a su manera.
#
# Esta función lo deja siempre como 5 dígitos limpios:
#    · Quita espacios de las orillas.
#    · Quita el ".0" que mete Excel al guardar números.
#    · Deja solo dígitos (fuera guiones, puntos, letras).
#    · Rellena con ceros a la izquierda hasta tener 5 dígitos.
#    · Convierte vacíos y "nan" en nulos reales.
def normalizar_cp(s):
    return (s.astype(str).str.strip()                        # 🧹 Quitar espacios
            .str.replace(r"\.0$", "", regex=True)             # 🔢 Quitar ".0" de Excel
            .str.replace(r"[^\d]", "", regex=True)            # ✂️ Solo dígitos
            .replace({"": np.nan, "nan": np.nan})             # 🕳️ Vacíos → NaN
            .str.zfill(5))                                    # 🧩 Rellenar a 5 dígitos

df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

print(f"\n🔑 CP normalizado en ambas tablas.")
print(f"  → '10115.0', ' 10115 ', '101-15' ahora todos son '10115'.")

# ------------------------------------------------------------
# 2. RENOMBRAR
# ------------------------------------------------------------
# 🏷️ Antes de cruzar, hay que evitar confusiones: ambos lados
#    tienen columnas que se llaman "state" pero significan cosas
#    distintas. Las "state" del cliente vienen de su dirección
#    libre; las "state" de Alemania vienen del catálogo oficial.
#
# Le ponemos prefijos según su origen:
#    · customer_  → datos que describen al cliente
#    · municipality_ → datos que describen al municipio oficial
#
# Es como poner etiquetas "de Juan" y "de María" a las cosas que
# se parecen, para no confundirlas después.
df_seguros = df_seguros.rename(columns={
    "city": "customer_city", "address": "customer_address",
    "state": "customer_state", "country": "customer_country",
})

df_alemania = df_alemania.rename(columns={
    "state":              "municipality_state",
    "district":           "municipality_district",
    "regional_type":      "municipality_regional_type",
    "population_total":   "municipality_population",
    "population_density": "municipality_density",
    "male":               "municipality_male",
    "female":             "municipality_female",
    "latitude":           "municipality_latitude",
    "longitude":          "municipality_longitude",
})

print(f"\n🏷️ Renombradas: 'state' → 'customer_state' (cliente) y 'state' → 'municipality_state' (catálogo).")
print(f"  → Ahora sabemos de dónde viene cada dato, sin ambigüedad.")

# ------------------------------------------------------------
# 3. NORMALIZAR ESTADOS ANTES DEL GROUPBY
# ------------------------------------------------------------
# 🗾 Aquí viene uno de los trucos importantes: los estados alemanes
#    aparecen escritos de mil formas en la tabla de seguros:
#      · En inglés: "Bavaria", "Saxony", "Hesse"...
#      · En alemán: "Bayern", "Sachsen", "Hessen"...
#      · Con abreviaturas ISO: "BY", "BW", "NW"...
#      · Y hasta con basura: "N/A", "-", "" ...
#
# Para poder cruzarlos con el catálogo alemán, todos deben
# convertirse al nombre oficial en alemán.
# Es como traducir todos los nombres de los estados al idioma
# del catálogo, para poder buscarlos.
mapeo_estados = {
    # Inglés → alemán
    "bavaria": "bayern", "saxony": "sachsen", "hesse": "hessen",
    "north rhine-westphalia": "nordrhein-westfalen",
    "nrw": "nordrhein-westfalen",
    "lower saxony": "niedersachsen",
    "rhineland-palatinate": "rheinland-pfalz",
    "thuringia": "thüringen",
    "saxony-anhalt": "sachsen-anhalt",
    "mecklenburg-western pomerania": "mecklenburg-vorpommern",
    "baden-wuerttemberg": "baden-württemberg",
    "baden-wurttemberg": "baden-württemberg",
    # Abreviaturas ISO
    "bw": "baden-württemberg", "by": "bayern", "be": "berlin",
    "bb": "brandenburg", "hb": "bremen", "hh": "hamburg",
    "he": "hessen", "mv": "mecklenburg-vorpommern",
    "ni": "niedersachsen", "nw": "nordrhein-westfalen",
    "rp": "rheinland-pfalz", "sl": "saarland",
    "sn": "sachsen", "st": "sachsen-anhalt",
    "sh": "schleswig-holstein", "th": "thüringen",
    # Alemán (con y sin tildes)
    "bayern": "bayern", "sachsen": "sachsen", "hessen": "hessen",
    "nordrhein-westfalen": "nordrhein-westfalen",
    "niedersachsen": "niedersachsen",
    "baden-württemberg": "baden-württemberg",
    "baden-wurttemberg": "baden-württemberg",
    "rheinland-pfalz": "rheinland-pfalz",
    "schleswig-holstein": "schleswig-holstein",
    "brandenburg": "brandenburg", "berlin": "berlin",
    "hamburg": "hamburg", "bremen": "bremen",
    "thüringen": "thüringen", "thuringen": "thüringen",
    "sachsen-anhalt": "sachsen-anhalt",
    "mecklenburg-vorpommern": "mecklenburg-vorpommern",
    "saarland": "saarland",
    # Basura (texto que no es un estado real)
    "n/a": None, "na": None, "-": None, "": None,
}

# Aplicamos el mapeo: minúsculas → sin espacios → traducido.
# Los que no estén en el diccionario quedan como NaN (nulos).
df_seguros["customer_state_norm"] = (
    df_seguros["customer_state"].str.lower().str.strip().map(mapeo_estados)
)

# 🚦 Semáforo de estados: cuántos quedaron sin traducir.
sin_mapear = df_seguros["customer_state_norm"].isna().sum()
pct_sin_mapear = sin_mapear / len(df_seguros) * 100

if   pct_sin_mapear == 0:   icono = "✅"  # Todos se tradujeron, increíble.
elif pct_sin_mapear < 5:    icono = "🟢"  # Casi todos, apenas unos pocos.
elif pct_sin_mapear < 20:   icono = "🟡"  # Aceptable, algunos huecos.
elif pct_sin_mapear < 50:   icono = "🟠"  # Bastantes sin traducir, ojo.
else:                        icono = "🔴"  # La mayoría sin traducir, algo falla.

print(f"\n{icono} Estados sin mapear: {sin_mapear:,} de {len(df_seguros):,} ({pct_sin_mapear:.1f}%)")
if pct_sin_mapear > 5:
    print(f"  → Esos son los estados que no están en el diccionario ni son")
    print(f"    basura reconocible. Habría que revisar si hay variantes nuevas.")

# ------------------------------------------------------------
# 4. CONVERTIR NUMÉRICOS EN ALEMANIA
# ------------------------------------------------------------
# 🔢 Como leímos todo como texto, las columnas numéricas de Alemania
#    quedaron como strings. Para poder sumar y promediar, hay que
#    convertirlas a número. También limpiamos comas decimales
#    europeas ("13,5" → "13.5") y las convertimos a flotante.
for col in ["municipality_population", "municipality_density",
            "municipality_male", "municipality_female",
            "municipality_latitude", "municipality_longitude"]:
    df_alemania[col] = pd.to_numeric(
        df_alemania[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

print(f"\n🔢 Columnas numéricas convertidas en Alemania (población, densidad, coords...)")
print(f"  → Ahora son números, listos para sumar y promediar.")

# ------------------------------------------------------------
# 5. MERGE POR CP (a nivel póliza, ANTES del groupby)
# ------------------------------------------------------------
# 🎯 Primer cruce: por código postal. Es el más preciso porque
#    une cada póliza con el municipio exacto del cliente.
#
# Como puede haber varios municipios con el mismo CP (raro pero
# pasa), agrupamos Alemania a 1 fila por CP:
#   · Textos (nombre, estado, distrito): "first" → el primero.
#   · Población: "sum" → los sumamos.
#   · Densidad, lat, lon: "mean" → promediamos.
#
# suffixes=("", "_cp") evita sufijos feos si ya había columnas
# con "municipality_" en la tabla de seguros (no debería, pero
# por si acaso, las del nuevo merge llevan "_cp").
df_alemania_cp = df_alemania.groupby("postal_code", as_index=False).agg({
    "municipality_name":       "first",
    "municipality_state":      "first",
    "municipality_district":   "first",
    "municipality_population": "sum",
    "municipality_density":    "mean",
    "municipality_latitude":   "mean",
    "municipality_longitude":  "mean",
})

df_seguros = pd.merge(
    df_seguros, df_alemania_cp,
    on="postal_code", how="left",
    validate="many_to_one",
    suffixes=("", "_cp"),   # ← evita colisión si ya había municipality_*
)

# 🚦 Semáforo del match por CP: cuántas pólizas encontraron su
#    municipio exacto usando el código postal.
match_cp = df_seguros["municipality_name"].notna().sum()
pct_cp = match_cp / len(df_seguros) * 100

if   pct_cp >= 95: icono_cp = "✅"
elif pct_cp >= 80: icono_cp = "🟢"
elif pct_cp >= 50: icono_cp = "🟡"
else:              icono_cp = "🔴"

print(f"\n{icono_cp} Merge por CP: {match_cp:,} / {len(df_seguros):,} ({pct_cp:.1f}%)")
print(f"  → Esas pólizas encontraron su municipio exacto por código postal.")

# ------------------------------------------------------------
# 6. MERGE POR ESTADO (a nivel póliza, ANTES del groupby)
# ------------------------------------------------------------
# 🎯 Segundo cruce: por estado. Este es el "plan B" para las
#    pólizas cuyo CP no está en el catálogo alemán. En vez de
#    dejarlas sin nada, les asignamos la población y densidad
#    del ESTADO donde viven.
#
# Es menos preciso que el CP, pero mejor que nada. Es como si
#    no encuentro la dirección exacta, al menos sé en qué ciudad vive.
#
# Agrupamos Alemania por estado: sumamos población, promediamos densidad.
# Renombramos las columnas con prefijo "state_" para que se
# distingan de las del municipio.
df_alemania_estado = df_alemania.groupby("municipality_state", as_index=False).agg({
    "municipality_population": "sum",
    "municipality_density":    "mean",
})

df_seguros = pd.merge(
    df_seguros,
    df_alemania_estado.rename(columns={
        "municipality_state":      "state_key",
        "municipality_population": "state_population",
        "municipality_density":    "state_density",
    }),
    left_on="customer_state_norm",   # Columna del cliente (ya traducida)
    right_on="state_key",             # Columna del catálogo alemán
    how="left",
    validate="many_to_one",
)

# 🚦 Semáforo del match por estado.
match_estado = df_seguros["state_key"].notna().sum()
pct_estado = match_estado / len(df_seguros) * 100

if   pct_estado >= 95: icono_estado = "✅"
elif pct_estado >= 80: icono_estado = "🟢"
elif pct_estado >= 50: icono_estado = "🟡"
else:                  icono_estado = "🔴"

print(f"{icono_estado} Merge por estado: {match_estado:,} / {len(df_seguros):,} ({pct_estado:.1f}%)")
print(f"  → Esas pólizas encontraron al menos su estado (plan B).")
print(f"  → Ideal: que CP + estado den cobertura completa para no perder info.")

# ------------------------------------------------------------
# 7. AHORA SÍ: AGREGAR POR policy_id
# ------------------------------------------------------------
# 🔽 Hasta aquí cada fila era un RECLAMO individual. Pero una misma
#    póliza puede tener varios reclamos (o ninguno). Queremos una
#    sola fila por póliza, con los reclamos resumidos.
#
# Es como pasar de una lista donde cada cliente aparece varias veces
#    (una por cada compra) a un resumen donde cada cliente aparece
#    UNA sola vez, pero con el total de sus compras.
#
# Para eso decidimos cómo agregar cada columna:
#   · Montos de reclamo → "sum" (sumamos todos los reclamos).
#   · fraud_score, risk_score → "max" (nos quedamos con el peor).
#   · Todo lo demás → "first" (tomamos el primer valor, es el mismo).
#
# Agregamos también "_n_claims" para contar cuántos reclamos tuvo
# cada póliza. Es como agregar una columna "N° de compras".
cols_agg = {}
# Estas columnas describen el reclamo individual, no queremos agregarlas
# con "first" porque perdemos información específica de cada uno.
excluir_agg = {"policy_id", "claim_id", "claim_date", "claim_status",
               "claim_description", "incident_date", "incident_location",
               "police_report_filed", "police_report_number",
               "adjuster_id", "adjuster_name", "adjustment_date",
               "settlement_date", "claim_denial_reason", "fraud_flag"}

for c in df_seguros.columns:
    if c == "policy_id":                              # La llave de agrupación
        continue
    if c in ("claim_amount_eur", "claim_approved_amount_eur", "settlement_amount_eur"):
        cols_agg[c] = "sum"                           # 💰 Sumar montos
    elif c in ("fraud_score", "risk_score"):
        cols_agg[c] = "max"                           # ⚠️ Peor score
    else:
        cols_agg[c] = "first"                         # 🌟 Cualquiera, son iguales

# Añadimos un contador de reclamos por póliza.
df_seguros["_n_claims"] = 1
cols_agg["_n_claims"] = "sum"

df_final = df_seguros.groupby("policy_id", as_index=False).agg(cols_agg)
# Renombramos el contador a algo más descriptivo.
df_final = df_final.rename(columns={"_n_claims": "num_claims_detail"})

print(f"\n🔽 Dataset final: {len(df_final):,} filas (1 por policy_id)")
print(f"  → Antes había {len(df_seguros):,} filas (una por reclamo).")
print(f"    Ahora hay {len(df_final):,} filas (una por póliza).")
print(f"    Los reclamos múltiples se sumaron o resumieron.")

# ------------------------------------------------------------
# 8. COMBINAR: usar municipio si existe, si no estado
# ------------------------------------------------------------
# 🔀 Ya tenemos dos fuentes de datos geográficos:
#    · municipality_population (viene del CP, más preciso).
#    · state_population (viene del estado, plan B).
#
# La lógica es: usamos el dato del municipio si existe. Si no,
# caemos al del estado. Es como preferir la dirección exacta,
# pero conformarnos con la ciudad si no la tenemos.
df_final["municipality_population"] = df_final["municipality_population"].fillna(
    df_final["state_population"]
)
df_final["municipality_density"] = df_final["municipality_density"].fillna(
    df_final["state_density"]
)

# Después de combinarlas, ya no necesitamos las columnas "state_"
# porque su información ya está mezclada en las "municipality_".
# Es como tirar la escalera después de subir al techo.
df_final = df_final.drop(columns=["state_population", "state_density", "state_key"])

print(f"\n🔀 Municipio y estado combinados:")
print(f"    Si hay dato de municipio, se usa ese. Si no, el del estado.")
print(f"    Las columnas 'state_population' y 'state_density' ya se eliminaron.")

# ------------------------------------------------------------
# 9. VERIFICACIÓN FINAL
# ------------------------------------------------------------
# 🧾 Repaso final antes de guardar. Es como el chef que prueba
#    el plato antes de sacarlo: nos aseguramos de que todo esté
#    en orden y nada se haya roto.
print("\n" + "-" * 60)
print("🧾 VERIFICACIÓN FINAL")
print("-" * 60)
print(f"Filas finales: {len(df_final):,}")
print(f"policy_id únicos: {df_final['policy_id'].nunique():,}")
print(f"Duplicados: {df_final['policy_id'].duplicated().sum()}")

# 🚦 Semáforo de duplicados: si hay, es que algo se rompió en el groupby.
dups = df_final['policy_id'].duplicated().sum()
if dups == 0:
    print(f"  ✅ Cada policy_id aparece una sola vez. Todo bien.")
else:
    print(f"  🔴 Hay {dups} duplicados. Algo falló en la agrupación.")

# 🚦 Cobertura geográfica: cuántas pólizas tienen cada nivel de dato.
print(f"\n🗺️  Cobertura geográfica:")
con_municipio = (df_final['municipality_name'].notna()).sum()
pct_mun = con_municipio / len(df_final) * 100

# Como ya eliminamos 'state_key', verificamos con customer_state_norm.
if 'customer_state_norm' in df_final.columns:
    con_estado = df_final['customer_state_norm'].notna().sum()
else:
    con_estado = df_final['state_key'].notna().sum() if 'state_key' in df_final.columns else 0
pct_est = con_estado / len(df_final) * 100

# Semáforo de cobertura.
if   pct_mun >= 95: icono_mun = "✅"
elif pct_mun >= 80: icono_mun = "🟢"
elif pct_mun >= 50: icono_mun = "🟡"
else:               icono_mun = "🔴"

if   pct_est >= 95: icono_est = "✅"
elif pct_est >= 80: icono_est = "🟢"
elif pct_est >= 50: icono_est = "🟡"
else:               icono_est = "🔴"

print(f"  {icono_mun} Con municipio (por CP): {con_municipio:,} ({pct_mun:.1f}%)")
print(f"  {icono_est} Con estado (plan B):     {con_estado:,} ({pct_est:.1f}%)")
print(f"  → Ideal: si un cliente no encontró su municipio, al menos")
print(f"    encontró su estado. Así ningún dato geográfico se pierde.")

# Verificación dura: si hay duplicados, explota con error.
# Es como un "candado de seguridad": si algo se rompió, mejor
# que se detenga aquí y no seguir con datos sucios.
assert df_final["policy_id"].duplicated().sum() == 0, "❌ Duplicados"
print(f"\n  ✅ Candado de seguridad superado: no hay duplicados de policy_id.")

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos la tabla enriquecida final. Es el resultado de todo
#    el trabajo: cada póliza con sus reclamos resumidos, más los
#    datos geográficos del municipio y/o estado del cliente.
df_final.to_csv("insurance_life_enriched.csv", index=False)
print(f"\n✅ Guardado: insurance_life_enriched.csv ({len(df_final):,} filas)")
print(f"  → Ahora cada póliza tiene: sus reclamos sumados, el municipio")
print(f"    (o estado) del cliente, y todos los datos limpios que")
print(f"    preparamos en los pasos anteriores. Lista para análisis.")
# ------------------------------------------------------------
# 11. BOTÓN DE DESCARGA (Google Colab)
# ------------------------------------------------------------
# 📥 Este botón permite bajar el CSV resultante con un clic,
#    en vez de tener que ir al panel de archivos de Colab.
from google.colab import files

print("\n" + "-" * 60)
print("📥 DESCARGAR RESULTADO")
print("-" * 60)
print("Hacé clic en el botón de abajo para descargar el CSV:")
files.download("insurance_life_enriched.csv")

📥 Cargado:
    Seguros:  6,593 filas x 69 columnas
    Alemania: 10,757 filas x 12 columnas

🔑 CP normalizado en ambas tablas.
  → '10115.0', ' 10115 ', '101-15' ahora todos son '10115'.

🏷️ Renombradas: 'state' → 'customer_state' (cliente) y 'state' → 'municipality_state' (catálogo).
  → Ahora sabemos de dónde viene cada dato, sin ambigüedad.

✅ Estados sin mapear: 0 de 6,593 (0.0%)

🔢 Columnas numéricas convertidas en Alemania (población, densidad, coords...)
  → Ahora son números, listos para sumar y promediar.

✅ Merge por CP: 6,593 / 6,593 (100.0%)
  → Esas pólizas encontraron su municipio exacto por código postal.
✅ Merge por estado: 6,593 / 6,593 (100.0%)
  → Esas pólizas encontraron al menos su estado (plan B).
  → Ideal: que CP + estado den cobertura completa para no perder info.

🔽 Dataset final: 6,512 filas (1 por policy_id)
  → Antes había 6,593 filas (una por reclamo).
    Ahora hay 6,512 filas (una por póliza).
    Los reclamos múltiples se sumaron o resumieron.

🔀 Munici

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>